# Translation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install evaluate sacrebleu transformers datasets evaluate sacrebleu accelerate sacremoses

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 21.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import evaluate
from sklearn.model_selection import train_test_split
import numpy as np

In [5]:
# df_translation = pd.read_csv("https://drive.google.com/uc?id=1ufEzybfWMXJh4W_AFxQp_c2mmGESRDT2")
# train_df, temp_df = train_test_split(df_translation, test_size=0.2, random_state=42)
# val_df, test_df = train_test_split(temp_df, test_size=(200/1200), random_state=42)
# train_df.to_csv('train.csv', index=False)
# val_df.to_csv('val.csv', index=False)
# test_df.to_csv('test.csv', index=False)

train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('val.csv')
test_df = pd.read_csv('test_merged.csv')

print(f"   Train Rows : {len(train_df)}")
print(f"   Val Rows   : {len(val_df)}")
print(f"   Test Rows  : {len(test_df)}")

   Train Rows : 4800
   Val Rows   : 1000
   Test Rows  : 200


In [6]:
# Calculate character length for each row
train_df['char_count'] = train_df['text'].astype(str).apply(len)

# Get Min and Max
min_chars = train_df['char_count'].min()
max_chars = train_df['char_count'].max()

print(f"Minimum Character Count: {min_chars}")
print(f"Maximum Character Count: {max_chars}")

# Show rows with Min and Max lengths
print("\n--- Shortest Text ---")
print(train_df[train_df['char_count'] == min_chars][['text', 'char_count']].head(1))

print("\n--- Longest Text ---")
print(train_df[train_df['char_count'] == max_chars][['text', 'char_count']].head(1))

# See full statistics (Mean, Std, Percentiles)
print("\n--- Statistics ---")
print(train_df['char_count'].describe())

Minimum Character Count: 31
Maximum Character Count: 343

--- Shortest Text ---
                                 text  char_count
1570  bagus bersih masih sepi jd enak          31

--- Longest Text ---
                                                   text  char_count
1681  datang ke sini di weekday enak banget ya bisa ...         343

--- Statistics ---
count    4800.000000
mean      150.130833
std        71.306882
min        31.000000
25%        91.000000
50%       133.000000
75%       199.000000
max       343.000000
Name: char_count, dtype: float64


In [7]:
class TranslationDataset(Dataset):
    def __init__(self, texts):
        self.texts = [str(t) if pd.notna(t) else "" for t in texts]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

In [ ]:
# --- OPTION B: NLLB (Meta - Higher Accuracy, Slower) ---
# MODEL_NAME = "facebook/nllb-200-distilled-600M"

# --- OPTION C: M2M100 (Meta - General Purpose) ---
# MODEL_NAME = "facebook/m2m100_418M"

In [8]:
def evaluate_translation(df, target_col, output_col, model_name):
    """
    Calculates and prints the SacreBLEU score for translation predictions.

    Args:
        df (pd.DataFrame): DataFrame containing predictions and ground truth.
        target_col (str): Name of the column with ground truth translations.
        output_col (str): Name of the column with model predictions.
        model_name (str): Name of the model for display purposes.
    """
    if target_col in df.columns:
        print("\nCalculating Evaluation Metrics...")
        bleu_metric = evaluate.load("sacrebleu")
        meteor_metric = evaluate.load("meteor")

        # Filter valid rows
        eval_df = df.dropna(subset=[target_col, output_col])

        preds = eval_df[output_col].tolist()
        refs = [[ref] for ref in eval_df[target_col].tolist()]

        if len(refs) > 0:
            bleu_results = bleu_metric.compute(predictions=preds, references=refs)
            meteor_results = meteor_metric.compute(predictions=preds, references=refs)
            print("-" * 30)
            print(f"Model: {model_name}")
            print(f"BLEU Score: {bleu_results['score']:.2f}")
            print(f"Meteor Score: {meteor_results['meteor']:.2f}")
            print("-" * 30)
        else:
            print("Ground truth column is empty.")
            return None
    else:
        print(f"Column '{target_col}' not found. Skipping evaluation.")
        return None

## Helsinki-NLP

In [ ]:
test_df = pd.read_csv('test_merged.csv')

### Baseline

In [ ]:
MODEL_NAME = "Helsinki-NLP/opus-mt-id-en"
BATCH_SIZE = 32
MAX_LENGTH = 128
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

def translate_batch(texts, model, tokenizer, device):
    inputs = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        translated = model.generate(**inputs, max_new_tokens=MAX_LENGTH, num_beams=4, early_stopping=True)

    return tokenizer.batch_decode(translated, skip_special_tokens=True)

source_texts = test_df['text'].tolist()

dataset = TranslationDataset(source_texts)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

all_predictions = []
for batch in tqdm(dataloader, desc="Translating Test Set"):
    if not isinstance(batch, list): batch = list(batch)
    preds = translate_batch(batch, model, tokenizer, device)
    all_predictions.extend(preds)

test_df['pred_helsinki_baseline'] = all_predictions

evaluate_translation(
    df=test_df,
    target_col='english_translation',
    output_col='pred_helsinki_baseline',
    model_name=f"{MODEL_NAME} (Baseline)"
)

evaluate_translation(
    df=test_df,
    target_col='better_english_translation',
    output_col='pred_helsinki_baseline',
    model_name=f"{MODEL_NAME} (Gold Standard)"
)

Translating Test Set: 100%|██████████| 7/7 [00:37<00:00,  5.40s/it]



Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: Helsinki-NLP/opus-mt-id-en (Baseline)
BLEU Score: 10.15
Meteor Score: 0.35
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: Helsinki-NLP/opus-mt-id-en (Baseline Gold Standard)
BLEU Score: 8.91
Meteor Score: 0.33
------------------------------


### Fine Tuning

In [ ]:
MODEL_CHECKPOINT = "Helsinki-NLP/opus-mt-id-en"
OUTPUT_DIR = "./fine_tuned_helsinki_tourism"
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

data_files_train = {
    "train": "train.csv",
    "validation": "val.csv"
}

dataset = load_dataset("csv", data_files=data_files_train)


tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess_function(examples):
    inputs = [str(ex) for ex in examples['text']]
    targets = [str(ex) for ex in examples['english_translation']]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

print(f"Loading model: {MODEL_CHECKPOINT}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    save_total_limit=2,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting Fine-Tuning...")
trainer.train()

print(f"Saving fine-tuned model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading model: Helsinki-NLP/opus-mt-id-en...


...

Starting Fine-Tuning...


Epoch,Training Loss,Validation Loss,Bleu
1,No log,1.374363,32.742882
2,1.621500,1.292307,34.537476
3,1.621500,1.270778,35.255728


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[54795]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Saving fine-tuned model to ./fine_tuned_helsinki_tourism...


('./fine_tuned_helsinki_tourism/tokenizer_config.json',
 './fine_tuned_helsinki_tourism/special_tokens_map.json',
 './fine_tuned_helsinki_tourism/vocab.json',
 './fine_tuned_helsinki_tourism/source.spm',
 './fine_tuned_helsinki_tourism/target.spm',
 './fine_tuned_helsinki_tourism/added_tokens.json')

In [ ]:
OUTPUT_DIR = "./fine_tuned_helsinki_tourism"
print("Loading Fine-Tuned Model for Evaluation...")
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to("cuda")
finetuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

def predict_finetuned(texts):
    inputs = finetuned_tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=128
    ).to("cuda")

    with torch.no_grad():
        translated = finetuned_model.generate(**inputs, max_new_tokens=128, num_beams=4, early_stopping=True)

    return finetuned_tokenizer.batch_decode(translated, skip_special_tokens=True)

print("Predicting on Test Set...")
test_texts = test_df['text'].tolist()
dataset = TranslationDataset(test_texts)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

all_preds = []
for batch in tqdm(dataloader):
    if not isinstance(batch, list): batch = list(batch)
    all_preds.extend(predict_finetuned(batch))

test_df['pred_helsinki_finetuned'] = all_preds

evaluate_translation(
    df=test_df,
    target_col='english_translation',
    output_col='pred_helsinki_finetuned',
    model_name="Helsinki Fine-Tuned"
)

evaluate_translation(
    df=test_df,
    target_col='better_english_translation',
    output_col='pred_helsinki_finetuned',
    model_name="Helsinki Fine-Tuned (Gold Standard)"
)

Loading Fine-Tuned Model for Evaluation...
Predicting on Test Set...


100%|██████████| 7/7 [00:27<00:00,  3.86s/it]



Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: Helsinki Fine-Tuned
BLEU Score: 33.11
Meteor Score: 0.65
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: Helsinki Fine-Tuned
BLEU Score: 28.20
Meteor Score: 0.60
------------------------------


In [ ]:
import shutil
import os
from google.colab import files

# Configuration
FOLDER_TO_ZIP = "./fine_tuned_helsinki_tourism"  # The folder you want to download
OUTPUT_FILENAME = "fine_tuned_helsinki_tourism"  # The name of the zip file

print(f"Zipping folder '{FOLDER_TO_ZIP}'...")

# Create the zip archive
shutil.make_archive(OUTPUT_FILENAME, 'zip', FOLDER_TO_ZIP)

print(f"Created {OUTPUT_FILENAME}.zip")
print("Starting download...")

# Trigger download
files.download(f"{OUTPUT_FILENAME}.zip")

Zipping folder './fine_tuned_helsinki_tourism'...
Created fine_tuned_helsinki_tourism.zip
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
test_df.to_csv('helsinki_results.csv', index=False)

## NLBB

In [9]:
test_df = pd.read_csv('test_merged.csv')

### Baseline

In [11]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "ind_Latn"
TGT_LANG = "eng_Latn"

BATCH_SIZE = 16
MAX_LENGTH = 128
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")
print(f"Loading model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

def translate_batch_nllb(texts, model, tokenizer, device):
    tokenizer.src_lang = SRC_LANG

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

    with torch.no_grad():
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_new_tokens=MAX_LENGTH,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

print(f"Running NLLB baseline on {len(test_df)} rows...")

dataset = TranslationDataset(test_df['text'].tolist())
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

all_predictions = []

for batch_texts in tqdm(dataloader, desc="Translating"):
    if not isinstance(batch_texts, list):
        batch_texts = list(batch_texts)

    batch_preds = translate_batch_nllb(batch_texts, model, tokenizer, device)
    all_predictions.extend(batch_preds)

test_df['pred_nllb_baseline'] = all_predictions

evaluate_translation(
    df=test_df,
    target_col='english_translation',
    output_col='pred_nllb_baseline',
    model_name="NLLB-200 Baseline"
)

evaluate_translation(
    df=test_df,
    target_col='better_english_translation',
    output_col='pred_nllb_baseline',
    model_name="NLLB-200 Baseline (Gold Standard)"
)

Using device: cuda
Loading model: facebook/nllb-200-distilled-600M...
Running NLLB baseline on 200 rows...


Translating: 100%|██████████| 13/13 [00:40<00:00,  3.15s/it]



Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: NLLB-200 Baseline
BLEU Score: 13.05
Meteor Score: 0.42
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: NLLB-200 Baseline (Gold Standard)
BLEU Score: 11.37
Meteor Score: 0.39
------------------------------


### Fine tuning

In [12]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR = "./fine_tuned_nllb_tourism"

BATCH_SIZE = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
MAX_LENGTH = 128

data_files = {
    "train": "train.csv",
    "validation": "val.csv",
}
dataset = load_dataset("csv", data_files=data_files)

print(f"Loading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.bos_token is None:
    tokenizer.bos_token = tokenizer.eos_token
tokenizer.src_lang = "ind_Latn"
tokenizer.tgt_lang = "eng_Latn"

def preprocess_nllb(examples):
    inputs = examples['text']
    targets = examples['english_translation']

    tokenizer.src_lang = SRC_LANG

    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=MAX_LENGTH,
        truncation=True
    )

    return model_inputs

print("Tokenizing data...")
tokenized_datasets = dataset.map(
    preprocess_nllb,
    batched=True
)

metric = evaluate.load("sacrebleu")

def compute_metrics_nllb(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

print(f"Loading model: {MODEL_NAME}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Force Target Language
forced_bos_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
model.config.forced_bos_token_id = forced_bos_id

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    save_total_limit=2,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    fp16=True if torch.cuda.is_available() else False,
    push_to_hub=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_nllb
)

print("Starting Fine-Tuning NLLB...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Loading tokenizer: facebook/nllb-200-distilled-600M...
Tokenizing data...


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading model: facebook/nllb-200-distilled-600M...
Starting Fine-Tuning NLLB...


Epoch,Training Loss,Validation Loss,Bleu
1,1.101800,0.869373,39.191125
2,0.884600,0.823505,40.735379
3,0.808000,0.816615,40.849036


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'forced_bos_token_id': 256047}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Model saved to ./fine_tuned_nllb_tourism


In [13]:
OUTPUT_DIR = "./fine_tuned_nllb_tourism"
TGT_LANG = "eng_Latn" # Target Language for NLLB (English)

print(f"Loading Fine-Tuned NLLB Model from {OUTPUT_DIR}...")
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to("cuda")
finetuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

forced_bos_id = finetuned_tokenizer.convert_tokens_to_ids(TGT_LANG)

def predict_finetuned_nllb(texts):
    inputs = finetuned_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to("cuda")

    with torch.no_grad():
        translated = finetuned_model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_id,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )

    return finetuned_tokenizer.batch_decode(translated, skip_special_tokens=True)

print("Predicting on Test Set (NLLB)...")
test_texts = test_df['text'].tolist()
dataset = TranslationDataset(test_texts)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

all_preds = []
for batch in tqdm(dataloader):
    if not isinstance(batch, list): batch = list(batch)
    all_preds.extend(predict_finetuned_nllb(batch))

# Save to specific column
test_df['pred_nllb_finetuned'] = all_preds

# Evaluate
evaluate_translation(
    df=test_df,
    target_col='english_translation',
    output_col='pred_nllb_finetuned',
    model_name="NLLB Fine-Tuned"
)

evaluate_translation(
    df=test_df,
    target_col='better_english_translation',
    output_col='pred_nllb_finetuned',
    model_name="NLLB Fine-Tuned (Gold Standard)"
)

Loading Fine-Tuned NLLB Model from ./fine_tuned_nllb_tourism...
Predicting on Test Set (NLLB)...


100%|██████████| 13/13 [00:56<00:00,  4.31s/it]



Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: NLLB Fine-Tuned
BLEU Score: 38.13
Meteor Score: 0.70
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: NLLB Fine-Tuned (Gold Standard)
BLEU Score: 32.37
Meteor Score: 0.64
------------------------------


In [20]:
test_df.to_csv('nllb_results.csv', index=False)

In [18]:
# import os
# import zipfile

# # Folder model kamu
# folder = "/content/fine_tuned_nllb_tourism"

# # File-file essential
# essential_files = [
#     "model.safetensors",
#     "config.json",
#     "sentencepiece.bpe.model",
#     "special_tokens_map.json",
#     "tokenizer.json",
#     "tokenizer_config.json"
# ]

# # Nama output ZIP
# zip_path = "/content/fine_tuned_nllb_tourism_essential.zip"

# # Buat ZIP
# with zipfile.ZipFile(zip_path, 'w') as z:
#     for fname in essential_files:
#         full_path = os.path.join(folder, fname)
#         if os.path.exists(full_path):
#             z.write(full_path, arcname=fname)
#         else:
#             print(f"File tidak ditemukan: {fname}")

# print("ZIP selesai dibuat:", zip_path)


ZIP selesai dibuat: /content/fine_tuned_nllb_tourism_essential.zip


In [19]:
# !mv fine_tuned_nllb_tourism_essential.zip /content/drive/MyDrive/